In [7]:
import pandas as pd
from ortools.linear_solver import pywraplp ##usado na ficha3

def read_data(filename, replace_values=[]):
    data = pd.read_csv(filename)
    for col, value in replace_values:
        data[col] = data[col].fillna(value)
    return data.values
horario = pywraplp.Solver.CreateSolver('SCIP')

periodos = 5
dias = ["Seg", "Ter", "Qua", "Qui", "Sex"]

salas = {}
sala_default = ""
for s in read_data("dados/salas.csv"):
    salas[s[0]] = s[2]
    if s[1] == "normal":
        sala_default = s[0]
##print(salas)

turmas = []
for t in read_data("dados/turmas.csv"):
    turmas.append(t[0]) 

disciplinas = {}
professores = {}
for d in read_data("dados/disciplinas.csv", [["sala_especial", sala_default]]):
    disciplinas[d[0]] = {
        "carga" : d[2], 
        "duplo": 1 if d[3] == "sim" else 0, 
        "professor": d[1], 
        "sala": d[4]}

    if d[1] not in professores:
        professores[d[1]] = {"disciplinas": [], "disponibilidade":{}}
    professores[d[1]]["disciplinas"].append(d[0])
#print(professores)
#print(disciplinas)

for disp in read_data("dados/disponibilidade_excecoes.csv"):
    if disp[1] not in professores[disp[0]]["disponibilidade"]:
        professores[disp[0]]["disponibilidade"][disp[1]] = []
    professores[disp[0]]["disponibilidade"][disp[1]].append(disp[2] - 1)

##print(professores)
x = {}
for t in turmas:
    x[t] = {}
    for d in dias:
        x[t][d] = {}
        for p in range(periodos):
            x[t][d][p] = {}
            for dis in disciplinas:
                x[t][d][p][dis] = horario.IntVar(0, 1, f"x_{t}_{d}_{p}_{dis}")

def get_Value(t,d,p,dis):
    return x[t][d][p][dis]

## R1. Uma turma não pode ter duas aulas em simultâneo.
for t in turmas:
    for d in dias:
        for p in range(periodos):
            horario.Add(sum(get_Value(t,d,p, dis) for dis in disciplinas) <= 1)

## R2. Cada disciplina cumpre *exatamente* a carga semanal definida em `disciplinas.csv`, para cada turma.
## print(disciplinas["Matemática"]) teste

for t in turmas:
    for dis in disciplinas:
        horario.Add(sum(get_Value(t, d, p, dis) for d in dias for p in range(periodos)) == disciplinas[dis]["carga"])

## R3. No máximo uma aula da mesma disciplina por dia, por turma — exceto disciplinas de duplo período (ver R4), em que o bloco de 2 tempos conta como uma só ocorrência nesse dia.
for t in turmas:
    for d in dias:
        for dis in disciplinas:
            if not disciplinas[dis]["duplo"]:
                horario.Add(sum(get_Value(t,d,p,dis) for p in range(periodos)) <= 1)
                
##R4. Disciplinas marcadas `duplo_periodo=sim` só podem ser dadas em blocos de 2 tempos consecutivos, no mesmo dia (nunca um tempo isolado).
for dis in disciplinas:
    if disciplinas[dis]["duplo"]:
        for t in turmas:
            for d in dias:
                ocorre = [get_Value(t,d,p,dis) for p in range(periodos)]
                horario.Add(sum(ocorre) <= 2)
                for p in range(periodos):
                    if p == 0:
                        horario.Add(get_Value(t,d,p,dis) <= get_Value(t,d,p+1,dis))
                    elif p == periodos-1:
                        horario.Add(get_Value(t,d,p,dis) <= get_Value(t,d,p-1,dis))
                    else:
                        horario.Add(get_Value(t,d,p,dis) <= get_Value(t,d,p-1,dis) + get_Value(t,d,p+1,dis));

##R5. Um professor não pode dar duas aulas em simultâneo, mesmo que sejam a turmas ou disciplinas diferentes.
for prof in professores:
        for d in dias:
            for p in range(periodos):
                horario.Add(sum(get_Value(t,d,p,dis) for t in turmas for dis in professores[prof]["disciplinas"] ) <= 1)

##R6. Um professor só pode dar aulas nos tempos em que está disponível (`disponibilidade_excecoes.csv`).
for prof in professores:
    for d in professores[prof]["disponibilidade"]:
        for p in professores[prof]["disponibilidade"][d]:
                horario.Add(sum(get_Value(t,d,p,dis) for t in turmas for dis in professores[prof]["disciplinas"]) == 0)

##R7. Cada aula ocupa uma sala. Disciplinas com `sala_especial` só podem usar salas desse tipo; as restantes usam salas `normal`. 
## Em nenhum tempo o número de aulas a decorrer num tipo de sala pode exceder a `quantidade` desse tipo definida em `salas.csv`.
for sala in salas:
    disciplinas_sala = [dis for dis in disciplinas if disciplinas[dis]["sala"] == sala]
    for d in dias:
        for p in range(periodos):
                horario.Add(sum(get_Value(t,d,p,dis) for t in turmas for dis in disciplinas_sala) <= salas[sala])
    

In [8]:
stat = horario.Solve()

if stat == pywraplp.Solver.OPTIMAL:
    for t in turmas:
        for d in dias:
            for p in range(periodos):
                for dis in disciplinas:
                    if int(get_Value(t,d,p,dis).solution_value()) == 1:
                        print("Solução encontrada!")
else:
    print("Sem Solução")

x_7ºA_Seg_0_Inglês
x_7ºA_Seg_1_Ciências
x_7ºA_Seg_3_Educação Física
x_7ºA_Seg_4_Educação Física
x_7ºA_Ter_0_Inglês
x_7ºA_Ter_3_Matemática
x_7ºA_Ter_4_Português
x_7ºA_Qua_0_Ciências
x_7ºA_Qua_3_Português
x_7ºA_Qua_4_Matemática
x_7ºA_Qui_0_Ciências
x_7ºA_Qui_2_História
x_7ºA_Qui_3_Português
x_7ºA_Qui_4_Matemática
x_7ºA_Sex_2_História
x_7ºA_Sex_3_Matemática
x_7ºA_Sex_4_Português
x_7ºB_Seg_0_Ciências
x_7ºB_Seg_1_Inglês
x_7ºB_Seg_3_Matemática
x_7ºB_Seg_4_Português
x_7ºB_Ter_1_Ciências
x_7ºB_Ter_2_História
x_7ºB_Ter_3_Educação Física
x_7ºB_Ter_4_Educação Física
x_7ºB_Qua_1_História
x_7ºB_Qua_2_Ciências
x_7ºB_Qua_3_Matemática
x_7ºB_Qua_4_Português
x_7ºB_Qui_0_Inglês
x_7ºB_Qui_3_Matemática
x_7ºB_Qui_4_Português
x_7ºB_Sex_3_Português
x_7ºB_Sex_4_Matemática


In [9]:
horas = [f"{8 + p}:00" for p in range(periodos)]

def apresentar_horario(turmas):
    for turma in turmas:
        print(f"\n\tTurma {turma}")
        print(f"{'':8}" + "".join(f"{d:^12}" for d in dias))
        for p in range(periodos):
            linha = f"{horas[p]:8}"
            for d in dias:
                coluna = "-"
                for dis in disciplinas:
                    if int(get_Value(turma, d, p, dis).solution_value()):
                        coluna = dis
                linha += f"{coluna:^12}"
            print(linha)

apresentar_horario(turmas)


	Turma 7ºA
            Seg         Ter         Qua         Qui         Sex     
8:00       Inglês      Inglês     Ciências    Ciências       -      
9:00      Ciências       -           -           -           -      
10:00        -           -           -        História    História  
11:00   Educação Física Matemática  Português   Português   Matemática 
12:00   Educação Física Português   Matemática  Matemática  Português  

	Turma 7ºB
            Seg         Ter         Qua         Qui         Sex     
8:00      Ciências       -           -         Inglês        -      
9:00       Inglês     Ciências    História       -           -      
10:00        -        História    Ciências       -           -      
11:00    Matemática Educação Física Matemática  Matemática  Português  
12:00    Português  Educação Física Português   Português   Matemática 
